In [1]:
import torch

print("PyTorch:", torch.__version__)
print("GPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU not available. Go to Runtime > Change runtime type > T4 GPU")

PyTorch: 2.11.0+cpu
GPU Available: False


In [2]:
!pip -q install fvcore openpyxl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 808.2 kB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 1.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 3.3 MB/s eta 0:00:00


In [3]:
import os
import zipfile
import time
import copy
import random
import shutil
import numpy as np
import pandas as pd

from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import torchvision
from torchvision import datasets, transforms, models
from torchvision.models import (
    AlexNet_Weights,
    VGG16_Weights,
    VGG19_Weights,
    ResNet18_Weights,
    ResNet50_Weights,
    ResNet101_Weights,
    DenseNet121_Weights,
    EfficientNet_B0_Weights
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC, SVC
from xgboost import XGBClassifier

from fvcore.nn import FlopCountAnalysis

import matplotlib.pyplot as plt

In [4]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [12]:
import os

from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = "/content/drive/MyDrive"

print("Searching for ZIP files...\n")

zip_files = []

for root, dirs, files in os.walk(DRIVE_ROOT):
    for file in files:
        if file.lower().endswith(".zip"):
            full_path = os.path.join(root, file)
            zip_files.append(full_path)

if len(zip_files) == 0:
    print("❌ Koi ZIP file nahi mili.")
else:
    print("✅ ZIP files found:\n")

    for i, path in enumerate(zip_files):
        size = os.path.getsize(path) / (1024**2)
        print(i, "->", path)
        print("   Size:", round(size, 2), "MB\n")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Searching for ZIP files...

✅ ZIP files found:

0 -> /content/drive/MyDrive/FA23-BAI-004 (3).zip
   Size: 0.0 MB

1 -> /content/drive/MyDrive/FA23_BAI_004_ANN_Assignment_03.zip
   Size: 0.4 MB

2 -> /content/drive/MyDrive/Classroom/CSC232-Information Security-BAI SEMESTER-4A/IS FINAL PROJECT By Atif Ali Shah & M.Bilal Butt (2).ipynb.zip
   Size: 0.06 MB

3 -> /content/drive/MyDrive/Classroom/CSC232-Information Security-BAI SEMESTER-4A/IS FINAL PROJECT By Atif Ali Shah & M.Bilal Butt (1).ipynb.zip
   Size: 0.06 MB

4 -> /content/drive/MyDrive/Classroom/CSC232-Information Security-BAI SEMESTER-4A/IS FINAL PROJECT By Atif Ali Shah & M.Bilal Butt.ipynb.zip
   Size: 0.06 MB

5 -> /content/drive/MyDrive/Classroom/CSC232-Information Security-BAI SEMESTER-4A/IS FINAL PROJECT By Atif Ali Shah(004) & M.Bilal Butt(042).ipynb.zip
   Size: 0.07 MB

6 -> /content/drive/MyD

In [24]:
import os

DATA_FOLDER = "/content/drive/MyDrive/KAGGLE SKIN DATASET "

ZIP_PATH = os.path.join(DATA_FOLDER, "archive (2).zip")

print("ZIP PATH:")
print(ZIP_PATH)

print("\nZIP exists:", os.path.exists(ZIP_PATH))

if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError("❌ Skin dataset ZIP nahi mili.")

print(
    "ZIP Size:",
    round(os.path.getsize(ZIP_PATH) / (1024**2), 2),
    "MB"
)

ZIP PATH:
/content/drive/MyDrive/KAGGLE SKIN DATASET /archive (2).zip

ZIP exists: True
ZIP Size: 785.63 MB


In [25]:
import zipfile

print("Checking ZIP contents...\n")

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    names = zip_ref.namelist()

print("Total ZIP entries:", len(names))

for name in names[:100]:
    print(name)

Checking ZIP contents...

Total ZIP entries: 2357
Skin cancer ISIC The International Skin Imaging Collaboration/Test/actinic keratosis/ISIC_0010512.jpg
Skin cancer ISIC The International Skin Imaging Collaboration/Test/actinic keratosis/ISIC_0010889.jpg
Skin cancer ISIC The International Skin Imaging Collaboration/Test/actinic keratosis/ISIC_0024468.jpg
Skin cancer ISIC The International Skin Imaging Collaboration/Test/actinic keratosis/ISIC_0024470.jpg
Skin cancer ISIC The International Skin Imaging Collaboration/Test/actinic keratosis/ISIC_0024511.jpg
Skin cancer ISIC The International Skin Imaging Collaboration/Test/actinic keratosis/ISIC_0024646.jpg
Skin cancer ISIC The International Skin Imaging Collaboration/Test/actinic keratosis/ISIC_0024654.jpg
Skin cancer ISIC The International Skin Imaging Collaboration/Test/actinic keratosis/ISIC_0024707.jpg
Skin cancer ISIC The International Skin Imaging Collaboration/Test/actinic keratosis/ISIC_0024763.jpg
Skin cancer ISIC The Internation

In [26]:
import os
import zipfile

EXTRACT_PATH = "/content/skin_dataset"

os.makedirs(EXTRACT_PATH, exist_ok=True)

print("Extracting dataset...")
print("Please wait... ZIP is around 786 MB.")

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)

print("\n✅ Extraction completed!")
print("Extracted location:")
print(EXTRACT_PATH)

Extracting dataset...
Please wait... ZIP is around 786 MB.

✅ Extraction completed!
Extracted location:
/content/skin_dataset


In [27]:
import os

IMAGE_EXTENSIONS = (
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
)

total_images = 0

print("Dataset structure:\n")

for root, dirs, files in os.walk(EXTRACT_PATH):

    image_count = sum(
        1
        for f in files
        if f.lower().endswith(IMAGE_EXTENSIONS)
    )

    if image_count > 0:

        print(
            root,
            "->",
            image_count,
            "images"
        )

        total_images += image_count

print("\n================================")
print("TOTAL IMAGES:", total_images)
print("================================")

Dataset structure:

/content/skin_dataset/Skin cancer ISIC The International Skin Imaging Collaboration/Train/squamous cell carcinoma -> 181 images
/content/skin_dataset/Skin cancer ISIC The International Skin Imaging Collaboration/Train/melanoma -> 438 images
/content/skin_dataset/Skin cancer ISIC The International Skin Imaging Collaboration/Train/dermatofibroma -> 95 images
/content/skin_dataset/Skin cancer ISIC The International Skin Imaging Collaboration/Train/vascular lesion -> 139 images
/content/skin_dataset/Skin cancer ISIC The International Skin Imaging Collaboration/Train/pigmented benign keratosis -> 462 images
/content/skin_dataset/Skin cancer ISIC The International Skin Imaging Collaboration/Train/nevus -> 357 images
/content/skin_dataset/Skin cancer ISIC The International Skin Imaging Collaboration/Train/actinic keratosis -> 114 images
/content/skin_dataset/Skin cancer ISIC The International Skin Imaging Collaboration/Train/basal cell carcinoma -> 376 images
/content/skin

In [28]:
import os

DATASET_ROOT = "/content/skin_dataset/Skin cancer ISIC The International Skin Imaging Collaboration"

TRAIN_DIR = os.path.join(DATASET_ROOT, "Train")
TEST_DIR = os.path.join(DATASET_ROOT, "Test")

print("TRAIN DIR:")
print(TRAIN_DIR)

print("\nTEST DIR:")
print(TEST_DIR)

classes = sorted([
    folder for folder in os.listdir(TRAIN_DIR)
    if os.path.isdir(os.path.join(TRAIN_DIR, folder))
])

print("\nClasses:")
for i, class_name in enumerate(classes):
    print(i, "->", class_name)

print("\nTotal Classes:", len(classes))

TRAIN DIR:
/content/skin_dataset/Skin cancer ISIC The International Skin Imaging Collaboration/Train

TEST DIR:
/content/skin_dataset/Skin cancer ISIC The International Skin Imaging Collaboration/Test

Classes:
0 -> actinic keratosis
1 -> basal cell carcinoma
2 -> dermatofibroma
3 -> melanoma
4 -> nevus
5 -> pigmented benign keratosis
6 -> seborrheic keratosis
7 -> squamous cell carcinoma
8 -> vascular lesion

Total Classes: 9


In [29]:
import os
import pandas as pd

IMAGE_EXTENSIONS = (
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
)

records = []

class_to_idx = {
    class_name: index
    for index, class_name in enumerate(classes)
}

for class_name in classes:

    class_path = os.path.join(TRAIN_DIR, class_name)

    for file_name in os.listdir(class_path):

        if file_name.lower().endswith(IMAGE_EXTENSIONS):

            records.append({
                "path": os.path.join(class_path, file_name),
                "class_name": class_name,
                "label": class_to_idx[class_name]
            })

train_df = pd.DataFrame(records)

print("Training images:", len(train_df))

print("\nClass Distribution:")
print(
    train_df["class_name"].value_counts()
)

Training images: 2239

Class Distribution:
class_name
pigmented benign keratosis    462
melanoma                      438
basal cell carcinoma          376
nevus                         357
squamous cell carcinoma       181
vascular lesion               139
actinic keratosis             114
dermatofibroma                 95
seborrheic keratosis           77
Name: count, dtype: int64


In [30]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    train_df,
    test_size=0.20,
    stratify=train_df["label"],
    random_state=42
)

print("Train images:", len(train_df))
print("Validation images:", len(val_df))

Train images: 1791
Validation images: 448


In [31]:
test_records = []

for class_name in classes:

    class_path = os.path.join(TEST_DIR, class_name)

    for file_name in os.listdir(class_path):

        if file_name.lower().endswith(IMAGE_EXTENSIONS):

            test_records.append({
                "path": os.path.join(class_path, file_name),
                "class_name": class_name,
                "label": class_to_idx[class_name]
            })

test_df = pd.DataFrame(test_records)

print("Test images:", len(test_df))

Test images: 118


In [32]:
print("================================")
print("FINAL DATA SPLIT")
print("================================")

print("Train      :", len(train_df))
print("Validation :", len(val_df))
print("Test       :", len(test_df))

print("\nTotal:")
print(
    len(train_df) +
    len(val_df) +
    len(test_df)
)

FINAL DATA SPLIT
Train      : 1791
Validation : 448
Test       : 118

Total:
2357


In [33]:
!pip -q install fvcore xgboost openpyxl

In [34]:
import os
import time
import copy
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

from PIL import Image

import torchvision
import torchvision.transforms as transforms
import torchvision.models as models

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cpu
Torchvision: 0.26.0+cpu
Device: cpu


In [35]:
IMAGE_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("Transforms ready.")

Transforms ready.


In [36]:
class SkinDataset(Dataset):

    def __init__(self, dataframe, transform=None):

        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        image = Image.open(
            row["path"]
        ).convert("RGB")

        label = int(row["label"])

        if self.transform:
            image = self.transform(image)

        return image, label

In [37]:
train_dataset = SkinDataset(
    train_df,
    transform=train_transform
)

val_dataset = SkinDataset(
    val_df,
    transform=val_test_transform
)

test_dataset = SkinDataset(
    test_df,
    transform=val_test_transform
)

print("Train dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Test dataset:", len(test_dataset))

Train dataset: 1791
Validation dataset: 448
Test dataset: 118


In [38]:
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("DataLoaders ready.")

print(
    "Train batches:",
    len(train_loader)
)

print(
    "Validation batches:",
    len(val_loader)
)

print(
    "Test batches:",
    len(test_loader)
)

DataLoaders ready.
Train batches: 56
Validation batches: 14
Test batches: 4


In [39]:
images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Labels shape:", labels.shape)
print("Labels:", labels[:10])

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Image batch shape: torch.Size([32, 3, 224, 224])
Labels shape: torch.Size([32])
Labels: tensor([8, 5, 6, 3, 8, 5, 3, 2, 1, 2])


In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("❌ GPU not available")

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [7]:
import os

print("CONTENT FOLDER STRUCTURE")
print("========================")

for root, dirs, files in os.walk("/content"):

    level = root.replace("/content", "").count(os.sep)

    if level <= 4:
        print(root)

CONTENT FOLDER STRUCTURE
/content
/content/.config
/content/.config/logs
/content/.config/logs/2026.09.04
/content/.config/configurations
/content/sample_data


In [8]:
from google.colab import drive
import os

drive.mount("/content/drive")

DATA_FOLDER = "/content/drive/MyDrive/KAGGLE SKIN DATASET "

ZIP_PATH = os.path.join(
    DATA_FOLDER,
    "archive (2).zip"
)

print("ZIP exists:", os.path.exists(ZIP_PATH))
print("ZIP path:", ZIP_PATH)

if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(
        "❌ archive (2).zip nahi mili."
    )

print(
    "ZIP size:",
    round(os.path.getsize(ZIP_PATH) / (1024**2), 2),
    "MB"
)

Mounted at /content/drive
ZIP exists: True
ZIP path: /content/drive/MyDrive/KAGGLE SKIN DATASET /archive (2).zip
ZIP size: 785.63 MB


In [9]:
import os
import zipfile

EXTRACT_PATH = "/content/skin_dataset"

os.makedirs(
    EXTRACT_PATH,
    exist_ok=True
)

print("Extracting dataset...")
print("Please wait...")

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)

print("\n✅ Dataset extracted successfully!")

Extracting dataset...
Please wait...

✅ Dataset extracted successfully!


In [10]:
import os

train_folders = []
test_folders = []

for root, dirs, files in os.walk(EXTRACT_PATH):

    folder_name = os.path.basename(root).strip().lower()

    if folder_name == "train":
        train_folders.append(root)

    if folder_name == "test":
        test_folders.append(root)

print("Train folders found:")
for path in train_folders:
    print(path)

print("\nTest folders found:")
for path in test_folders:
    print(path)

if not train_folders:
    raise FileNotFoundError(
        "❌ Train folder nahi mila."
    )

if not test_folders:
    raise FileNotFoundError(
        "❌ Test folder nahi mila."
    )

TRAIN_DIR = train_folders[0]
TEST_DIR = test_folders[0]

print("\n==============================")
print("TRAIN_DIR:")
print(TRAIN_DIR)

print("\nTEST_DIR:")
print(TEST_DIR)

Train folders found:
/content/skin_dataset/Skin cancer ISIC The International Skin Imaging Collaboration/Train

Test folders found:
/content/skin_dataset/Skin cancer ISIC The International Skin Imaging Collaboration/Test

TRAIN_DIR:
/content/skin_dataset/Skin cancer ISIC The International Skin Imaging Collaboration/Train

TEST_DIR:
/content/skin_dataset/Skin cancer ISIC The International Skin Imaging Collaboration/Test


In [11]:
import os

IMAGE_EXTENSIONS = (
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
)

classes = sorted([
    folder
    for folder in os.listdir(TRAIN_DIR)
    if os.path.isdir(
        os.path.join(TRAIN_DIR, folder)
    )
])

class_to_idx = {
    class_name: i
    for i, class_name in enumerate(classes)
}

print("Classes:\n")

for i, class_name in enumerate(classes):
    print(i, "->", class_name)

print("\nTotal Classes:", len(classes))

Classes:

0 -> actinic keratosis
1 -> basal cell carcinoma
2 -> dermatofibroma
3 -> melanoma
4 -> nevus
5 -> pigmented benign keratosis
6 -> seborrheic keratosis
7 -> squamous cell carcinoma
8 -> vascular lesion

Total Classes: 9


In [12]:
import pandas as pd
from sklearn.model_selection import train_test_split

records = []

for class_name in classes:

    class_path = os.path.join(
        TRAIN_DIR,
        class_name
    )

    for file_name in os.listdir(class_path):

        if file_name.lower().endswith(
            IMAGE_EXTENSIONS
        ):

            records.append({
                "path": os.path.join(
                    class_path,
                    file_name
                ),
                "class_name": class_name,
                "label": class_to_idx[class_name]
            })

full_train_df = pd.DataFrame(records)

train_df, val_df = train_test_split(
    full_train_df,
    test_size=0.20,
    stratify=full_train_df["label"],
    random_state=42
)

test_records = []

for class_name in classes:

    class_path = os.path.join(
        TEST_DIR,
        class_name
    )

    for file_name in os.listdir(class_path):

        if file_name.lower().endswith(
            IMAGE_EXTENSIONS
        ):

            test_records.append({
                "path": os.path.join(
                    class_path,
                    file_name
                ),
                "class_name": class_name,
                "label": class_to_idx[class_name]
            })

test_df = pd.DataFrame(test_records)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))
print("Total:", len(train_df) + len(val_df) + len(test_df))

Train: 1791
Validation: 448
Test: 118
Total: 2357


In [13]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("PyTorch:", torch.__version__)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ GPU available nahi hai")

PyTorch: 2.11.0+cu128
Device: cuda
GPU: Tesla T4


In [14]:
from torchvision import transforms

IMAGE_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("✅ Transforms ready")

✅ Transforms ready


In [15]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image

class SkinDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        image = Image.open(
            row["path"]
        ).convert("RGB")

        label = int(row["label"])

        if self.transform:
            image = self.transform(image)

        return image, label

In [16]:
train_dataset = SkinDataset(
    train_df,
    train_transform
)

val_dataset = SkinDataset(
    val_df,
    val_test_transform
)

test_dataset = SkinDataset(
    test_df,
    val_test_transform
)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

Train: 1791
Validation: 448
Test: 118


In [17]:
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Train batches: 56
Validation batches: 14
Test batches: 4


In [18]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(classes)),
    y=train_df["label"].values
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
).to(device)

print("Class weights:\n")

for i, class_name in enumerate(classes):
    print(
        class_name,
        "->",
        round(class_weights[i].item(), 4)
    )

Class weights:

actinic keratosis -> 2.1868
basal cell carcinoma -> 0.6611
dermatofibroma -> 2.6184
melanoma -> 0.5686
nevus -> 0.6958
pigmented benign keratosis -> 0.5393
seborrheic keratosis -> 3.2097
squamous cell carcinoma -> 1.3724
vascular lesion -> 1.7928


In [19]:
images, labels = next(
    iter(train_loader)
)

print("Image shape:", images.shape)
print("Labels shape:", labels.shape)
print("Device:", device)

Image shape: torch.Size([32, 3, 224, 224])
Labels shape: torch.Size([32])
Device: cuda


In [20]:
import torch.nn as nn
import torchvision.models as models

from torchvision.models import (
    AlexNet_Weights,
    VGG16_Weights,
    VGG19_Weights,
    ResNet18_Weights,
    ResNet50_Weights,
    ResNet101_Weights,
    DenseNet121_Weights,
    EfficientNet_B0_Weights
)

NUM_CLASSES = len(classes)

print("Number of classes:", NUM_CLASSES)

Number of classes: 9


In [21]:
def create_model(model_name):

    if model_name == "AlexNet":

        model = models.alexnet(
            weights=AlexNet_Weights.DEFAULT
        )

        model.classifier[6] = nn.Linear(
            model.classifier[6].in_features,
            NUM_CLASSES
        )

    elif model_name == "VGG16":

        model = models.vgg16(
            weights=VGG16_Weights.DEFAULT
        )

        model.classifier[6] = nn.Linear(
            model.classifier[6].in_features,
            NUM_CLASSES
        )

    elif model_name == "VGG19":

        model = models.vgg19(
            weights=VGG19_Weights.DEFAULT
        )

        model.classifier[6] = nn.Linear(
            model.classifier[6].in_features,
            NUM_CLASSES
        )

    elif model_name == "ResNet18":

        model = models.resnet18(
            weights=ResNet18_Weights.DEFAULT
        )

        model.fc = nn.Linear(
            model.fc.in_features,
            NUM_CLASSES
        )

    elif model_name == "ResNet50":

        model = models.resnet50(
            weights=ResNet50_Weights.DEFAULT
        )

        model.fc = nn.Linear(
            model.fc.in_features,
            NUM_CLASSES
        )

    elif model_name == "ResNet101":

        model = models.resnet101(
            weights=ResNet101_Weights.DEFAULT
        )

        model.fc = nn.Linear(
            model.fc.in_features,
            NUM_CLASSES
        )

    elif model_name == "DenseNet121":

        model = models.densenet121(
            weights=DenseNet121_Weights.DEFAULT
        )

        model.classifier = nn.Linear(
            model.classifier.in_features,
            NUM_CLASSES
        )

    elif model_name == "EfficientNet-B0":

        model = models.efficientnet_b0(
            weights=EfficientNet_B0_Weights.DEFAULT
        )

        model.classifier[1] = nn.Linear(
            model.classifier[1].in_features,
            NUM_CLASSES
        )

    else:
        raise ValueError(
            "Unknown model: " + model_name
        )

    return model

In [22]:
MODEL_NAMES = [
    "AlexNet",
    "VGG16",
    "VGG19",
    "ResNet18",
    "ResNet50",
    "ResNet101",
    "DenseNet121",
    "EfficientNet-B0"
]

print("Total models:", len(MODEL_NAMES))

for model_name in MODEL_NAMES:
    print(model_name)

Total models: 8
AlexNet
VGG16
VGG19
ResNet18
ResNet50
ResNet101
DenseNet121
EfficientNet-B0


In [23]:
for model_name in MODEL_NAMES:

    model = create_model(model_name)

    total_params = sum(
        p.numel()
        for p in model.parameters()
    )

    print(
        model_name,
        "->",
        round(total_params / 1e6, 2),
        "M parameters"
    )

Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 176MB/s]


AlexNet -> 57.04 M parameters
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:07<00:00, 75.7MB/s]


VGG16 -> 134.3 M parameters
Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth


...

ResNet101 -> 42.52 M parameters
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 124MB/s]


DenseNet121 -> 6.96 M parameters
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 125MB/s]


EfficientNet-B0 -> 4.02 M parameters


In [24]:
import numpy as np
import torch

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

def evaluate_model(model, loader):

    model.eval()

    all_labels = []
    all_predictions = []
    all_probabilities = []

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(
                device,
                non_blocking=True
            )

            outputs = model(images)

            probabilities = torch.softmax(
                outputs,
                dim=1
            )

            predictions = probabilities.argmax(
                dim=1
            )

            all_labels.extend(
                labels.numpy()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

            all_probabilities.extend(
                probabilities.cpu().numpy()
            )

    y_true = np.array(all_labels)
    y_pred = np.array(all_predictions)
    y_prob = np.array(all_probabilities)

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    precision = precision_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    try:

        auc = roc_auc_score(
            y_true,
            y_prob,
            multi_class="ovr",
            average="macro"
        )

    except:

        auc = np.nan

    return {
        "Accuracy (%)": accuracy * 100,
        "Precision (%)": precision * 100,
        "Recall (%)": recall * 100,
        "F1-Score (%)": f1 * 100,
        "AUC (%)": auc * 100
    }

In [25]:
def train_model(
    model,
    train_loader,
    val_loader,
    epochs=5
):

    model = model.to(device)

    criterion = nn.CrossEntropyLoss(
        weight=class_weights
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=0.0001,
        weight_decay=0.0001
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=1
    )

    best_val_loss = float("inf")
    best_model_state = copy.deepcopy(
        model.state_dict()
    )

    for epoch in range(epochs):

        model.train()

        running_loss = 0.0
        correct = 0
        total = 0

        start_time = time.time()

        for images, labels in train_loader:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            loss.backward()

            optimizer.step()

            running_loss += (
                loss.item()
                * images.size(0)
            )

            predictions = outputs.argmax(
                dim=1
            )

            correct += (
                predictions == labels
            ).sum().item()

            total += labels.size(0)

        train_loss = (
            running_loss / total
        )

        train_accuracy = (
            correct / total
        ) * 100

        model.eval()

        val_loss_total = 0.0
        val_total = 0

        with torch.no_grad():

            for images, labels in val_loader:

                images = images.to(
                    device,
                    non_blocking=True
                )

                labels = labels.to(
                    device,
                    non_blocking=True
                )

                outputs = model(images)

                loss = criterion(
                    outputs,
                    labels
                )

                val_loss_total += (
                    loss.item()
                    * images.size(0)
                )

                val_total += labels.size(0)

        val_loss = (
            val_loss_total / val_total
        )

        scheduler.step(val_loss)

        if val_loss < best_val_loss:

            best_val_loss = val_loss

            best_model_state = copy.deepcopy(
                model.state_dict()
            )

        elapsed = time.time() - start_time

        print(
            f"Epoch {epoch + 1}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_accuracy:.2f}% | "
            f"Val Loss: {val_loss:.4f} | "
            f"Time: {elapsed:.1f}s"
        )

    model.load_state_dict(
        best_model_state
    )

    return model

In [27]:
import copy
import time

In [28]:
import copy
import time

def train_model(model, train_loader, val_loader, epochs=5):
    model = model.to(device)

    criterion = nn.CrossEntropyLoss(weight=class_weights)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=0.0001,
        weight_decay=0.0001
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=1
    )

    best_val_loss = float("inf")
    best_model_state = copy.deepcopy(model.state_dict())

    for epoch in range(epochs):

        model.train()

        running_loss = 0.0
        correct = 0
        total = 0

        start_time = time.time()

        for images, labels in train_loader:

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)

            predictions = outputs.argmax(dim=1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / total
        train_accuracy = (correct / total) * 100

        model.eval()

        val_loss_total = 0.0
        val_total = 0

        with torch.no_grad():

            for images, labels in val_loader:

                images = images.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)

                outputs = model(images)

                loss = criterion(outputs, labels)

                val_loss_total += loss.item() * images.size(0)
                val_total += labels.size(0)

        val_loss = val_loss_total / val_total

        scheduler.step(val_loss)

        if val_loss < best_val_loss:

            best_val_loss = val_loss

            best_model_state = copy.deepcopy(
                model.state_dict()
            )

        elapsed = time.time() - start_time

        print(
            f"Epoch {epoch+1}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_accuracy:.2f}% | "
            f"Val Loss: {val_loss:.4f} | "
            f"Time: {elapsed:.1f}s"
        )

    model.load_state_dict(best_model_state)

    return model

In [29]:
model = create_model("ResNet18")

print("Training ResNet18...")
print("Device:", device)

model = train_model(
    model,
    train_loader,
    val_loader,
    epochs=5
)

print("\n✅ ResNet18 training completed!")

Training ResNet18...
Device: cuda
Epoch 1/5 | Train Loss: 1.5206 | Train Acc: 39.64% | Val Loss: 1.0953 | Time: 37.1s
Epoch 2/5 | Train Loss: 0.9555 | Train Acc: 60.30% | Val Loss: 0.9986 | Time: 32.3s
Epoch 3/5 | Train Loss: 0.7966 | Train Acc: 65.72% | Val Loss: 0.9790 | Time: 33.9s
Epoch 4/5 | Train Loss: 0.6821 | Train Acc: 70.46% | Val Loss: 0.9208 | Time: 33.1s
Epoch 5/5 | Train Loss: 0.6117 | Train Acc: 73.37% | Val Loss: 0.9519 | Time: 31.7s

✅ ResNet18 training completed!


In [30]:
resnet18_results = evaluate_model(model, test_loader)

print("ResNet18 Test Results")

for metric, value in resnet18_results.items():
    print(metric, ":", round(value, 2))

ResNet18 Test Results
Accuracy (%) : 58.47
Precision (%) : 57.9
Recall (%) : 59.95
F1-Score (%) : 56.04
AUC (%) : 91.74


In [31]:
all_results = {}
trained_models = {}

for model_name in MODEL_NAMES:

    print("\n" + "="*60)
    print("Training:", model_name)
    print("="*60)

    model = create_model(model_name)

    model = train_model(
        model,
        train_loader,
        val_loader,
        epochs=5
    )

    results = evaluate_model(model, test_loader)

    all_results[model_name] = results
    trained_models[model_name] = model

    print("\nTest Results:")

    for metric, value in results.items():
        print(metric, ":", round(value, 2))

    del model
    torch.cuda.empty_cache()


Training: AlexNet
Epoch 1/5 | Train Loss: 1.6641 | Train Acc: 36.57% | Val Loss: 1.4282 | Time: 32.3s
Epoch 2/5 | Train Loss: 1.2845 | Train Acc: 50.92% | Val Loss: 1.2629 | Time: 32.3s
Epoch 3/5 | Train Loss: 1.0995 | Train Acc: 55.50% | Val Loss: 1.1688 | Time: 32.8s
Epoch 4/5 | Train Loss: 1.0329 | Train Acc: 58.63% | Val Loss: 1.3281 | Time: 31.8s
Epoch 5/5 | Train Loss: 0.9645 | Train Acc: 60.52% | Val Loss: 1.0830 | Time: 32.2s

Test Results:
Accuracy (%) : 50.85
Precision (%) : 52.71
Recall (%) : 59.72
F1-Score (%) : 52.06
AUC (%) : 89.05

Training: VGG16
Epoch 1/5 | Train Loss: 2.0011 | Train Acc: 24.90% | Val Loss: 1.5239 | Time: 43.4s
Epoch 2/5 | Train Loss: 1.6137 | Train Acc: 33.89% | Val Loss: 1.3266 | Time: 41.8s
Epoch 3/5 | Train Loss: 1.3375 | Train Acc: 45.51% | Val Loss: 1.1912 | Time: 43.0s
Epoch 4/5 | Train Loss: 1.2053 | Train Acc: 48.97% | Val Loss: 1.1534 | Time: 43.0s
Epoch 5/5 | Train Loss: 1.1551 | Train Acc: 53.88% | Val Loss: 1.1136 | Time: 42.8s

Test Resu